# Data Ingestion from heizoel24.ch

This notebook fetches the daily heating oil price dataset from the heizoel24.ch chart API and saves it to S3 for further processing.

The [heizoel24.ch chart page](https://www.heizoel24.ch/charts/heizoel) loads its data from an **undocumented JSON API** (reverse-engineered from the site's JS bundles). It provides **daily** heating oil prices.

- **Endpoint:** `GET https://www.heizoel24.ch/api/site/{countryId}/prices/history?amount=3000&productId=1&rangeType={N}`
- **countryId:** `1` = CH, `2` = DE
- **productId:** `1` = Heizöl (heating oil)
- **rangeType:** `4` = 1 month, `1` = 3 months, `3` = 6 months, `6` = 1 year, `7` = 3 years, `8` = 5 years, `9` = 10 years, `10` = year-to-date, `11` = full history (since 2003)
- **Response:** daily rows `{"DateTime": "2026-01-01T00:00:00", "Price": 164.9, "Demand": 11.7}` — price in **CHF per 100 l**

Caveat: this is an unofficial, undocumented API and may change without notice.

**Production ingestion:** the full history (7,028 rows, 2003-01-06 to today) is stored as the training data at `raw/oil_data.csv` (columns `date`, `price` = CHF/100 l, `demand`) in bucket `oilprice-features`. It is appended daily by `src/collect_daily_prices.py` (k8s CronJob `collect-daily-prices` at 23:30, before the feature and training pipelines), which auto-bootstraps the full history on cold start. The cells below show a manual one-off fetch of the same data.


In [12]:
import requests
import pandas as pd

# Fetch the full daily price history from the heizoel24.ch chart API
# countryId: 1 = CH | productId: 1 = heating oil | rangeType: 11 = full history (since 2003)
url = "https://www.heizoel24.ch/api/site/1/prices/history"
params = {"amount": 3000, "productId": 1, "rangeType": 11}
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, params=params, headers=headers, timeout=60)
if response.status_code != 200:
    raise Exception(f"API request failed with status code {response.status_code}")

df = pd.DataFrame(response.json())
df["date"] = pd.to_datetime(df["DateTime"]).dt.date.astype(str)
df = df[["date", "Price", "Demand"]].rename(columns={"Price": "price", "Demand": "demand"})
df = df.sort_values("date").drop_duplicates(subset="date").reset_index(drop=True)

print(f"Fetched {len(df)} daily records from heizoel24.ch ({df['date'].iloc[0]} to {df['date'].iloc[-1]})")
df.tail()


Fetched 7030 daily records from heizoel24.ch (2003-01-06 to 2026-10-04)


,date,price,demand
7025,2026-09-30,167.87,16.310204
7026,2026-10-01,168.63,15.755102
7027,2026-10-02,164.14,23.885714
7028,2026-10-03,163.99,10.987755
7029,2026-10-04,163.92,4.195918


In [2]:
import s3fs
import base64
import json
import subprocess

# read the S3 credentials from the Rook-generated secret in the rook-ceph namespace
out = subprocess.run(
    [
        "kubectl",
        "-n",
        "rook-ceph",
        "get",
        "secret",
        "rook-ceph-object-user-objectstore-objectstore-user",
        "-o",
        "json",
    ],
    capture_output=True,
    text=True,
    check=True,
).stdout
secret_data = json.loads(out)["data"]

# the values in the secret are base64-encoded
access_key = base64.b64decode(secret_data["AccessKey"]).decode()
secret_key = base64.b64decode(secret_data["SecretKey"]).decode()
endpoint = base64.b64decode(secret_data["Endpoint"]).decode()

print("S3 endpoint:", endpoint)

# s3fs does NOT read FSSPEC_S3_* environment variables: S3FileSystem() only
# accepts key/secret/endpoint_url, otherwise it falls back to botocore's standard
# AWS credential chain (AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY), which finds
# nothing here and raises NoCredentialsError. So pass them explicitly:
# https://s3fs.readthedocs.io/en/latest/index.html#s3-compatible-storage

s3_options = {
    "key": access_key,
    "secret": secret_key,
    "endpoint_url": endpoint,
}

s3 = s3fs.S3FileSystem(**s3_options)
if not s3.exists("traindata"):
    s3.mkdir("traindata")


S3 endpoint: http://rook-ceph-rgw-objectstore.rook-ceph.svc:8080


In [4]:
# Save to S3 (same key as the production ingestion in src/collect_daily_prices.py)
bucket_name = "oilprice-features"
key = "raw/oil_data.csv"

df.to_csv(f"s3://{bucket_name}/{key}", index=False, storage_options=s3_options)

print(f"Data saved to s3://{bucket_name}/{key}")


Data saved to s3://oilprice-features/raw/oil_data.csv


In [11]:
# Verify by reading from S3
check = pd.read_csv(f"s3://{bucket_name}/{key}", storage_options=s3_options)
print(f"shape: {check.shape}, columns: {check.columns.tolist()}")
print(f"date range: {check['date'].min()} .. {check['date'].max()}")
check.tail()


shape: (7030, 3), columns: ['date', 'price', 'demand']
date range: 2003-01-06 .. 2026-10-04


,date,price,demand
7025,2026-09-30,167.87,16.310204
7026,2026-10-01,168.63,15.755102
7027,2026-10-02,164.14,23.885714
7028,2026-10-03,163.99,10.987755
7029,2026-10-04,163.92,4.081633


## Training Data as Parquet (S3 `traindata` bucket)

The raw training data (full heizoel24.ch daily history) is stored as one Parquet
file in the `traindata` bucket: `s3://traindata/train_oilprice_raw.parquet`.

- **Bootstrap (once):** upload the full local history from
  `data/heizoel24_heizoel_daily_ch_full.csv` - or run `python src/upload_train_data.py`.
- **Daily update:** no cron job is needed. To keep the data up to date, append the
  single record of the day - run `python src/upload_train_data.py --append` or
  execute the last cell of this section.

Note: s3fs does **not** read `FSSPEC_S3_*` environment variables, so the
key/secret/endpoint_url from the Rook secret are passed explicitly via
`storage_options`.


In [13]:
# Bootstrap (once): upload the full local history as Parquet
# (same as `python src/upload_train_data.py`; run from the repo root or notebooks/)
import os
csv_path = os.path.join("data", "heizoel24_heizoel_daily_ch_full.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join("..", "data", "heizoel24_heizoel_daily_ch_full.csv")
df = pd.read_csv(csv_path)
df.to_parquet("s3://traindata/train_oilprice_raw.parquet", storage_options=s3_options)
print(f"uploaded s3://traindata/train_oilprice_raw.parquet: {len(df)} rows ({df['date'].iloc[0]} .. {df['date'].iloc[-1]})")


uploaded s3://traindata/train_oilprice_raw.parquet: 7028 rows (2003-01-06 .. 2026-10-02)


In [16]:
# verify: read the Parquet file back from S3
check = pd.read_parquet("s3://traindata/train_oilprice_raw.parquet", storage_options=s3_options)
print(f"shape: {check.shape}, columns: {check.columns.tolist()}")
print(f"date range: {check['date'].min()} .. {check['date'].max()}")
check.tail()


shape: (7030, 3), columns: ['date', 'price', 'demand']
date range: 2003-01-06 .. 2026-10-04


,date,price,demand
7025,2026-09-30,167.87,16.310204
7026,2026-10-01,168.63,15.755102
7027,2026-10-02,164.14,23.885714
7028,2026-10-03,163.99,10.987755
7029,2026-10-04,163.92,4.195918


### Daily update (no cron job)

No daily pipeline is needed for the training data. Whenever you want it up to
date, append the single record of the day (or all missing dates if it is run
late). The API call fetches the last 3 months, but only dates that are not in
the Parquet file yet (or changed) are added. The same logic lives in
`src/upload_train_data.py --append`.


In [15]:
# append the record(s) of the day (idempotent - safe to re-run)
response = requests.get(
    "https://www.heizoel24.ch/api/site/1/prices/history",
    params={"amount": 3000, "productId": 1, "rangeType": 1},
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30,
)
response.raise_for_status()
new = pd.DataFrame(response.json())
new["date"] = pd.to_datetime(new["DateTime"]).dt.date.astype(str)
new = new[["date", "Price", "Demand"]].rename(columns={"Price": "price", "Demand": "demand"})

existing = pd.read_parquet("s3://traindata/train_oilprice_raw.parquet", storage_options=s3_options)
merged = (
    pd.concat([existing, new], ignore_index=True)
    .drop_duplicates(subset="date", keep="last")
    .sort_values("date")
    .reset_index(drop=True)
)
merged.to_parquet("s3://traindata/train_oilprice_raw.parquet", storage_options=s3_options)
print(f"updated s3://traindata/train_oilprice_raw.parquet: {len(merged)} rows "
      f"({len(merged) - len(existing)} new record(s), latest: {merged['date'].iloc[-1]})")


updated s3://traindata/train_oilprice_raw.parquet: 7030 rows (2 new record(s), latest: 2026-10-04)
